# MODELO COLABORATIVO

El modelo utilizará las interacciones históricas usuario-película almacenadas en la capa Gold ML para aprender patrones de preferencias y generar recomendaciones Top-N personalizadas.

## 1. IMPORTACIONES

In [17]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix

from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)


## 2. CONFIGURACION DEL PROYECTO

In [18]:
PROJECT_ROOT = Path.cwd()

while (
    not (PROJECT_ROOT / "src").exists()
    and PROJECT_ROOT.parent != PROJECT_ROOT
):
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.utils.paths import (
    GOLD_ML_COLLABORATIVE_DIR,
    GOLD_DIMENSIONS_DIR,
)

print("PROJECT_ROOT:")
print(PROJECT_ROOT)

print("\nGOLD_ML_COLLABORATIVE_DIR:")
print(GOLD_ML_COLLABORATIVE_DIR)

PROJECT_ROOT:
d:\INSTITUTO CONTINENTAL\5. CICLO - 5\PROYECTO PRODUCTIVO IIIA\PP_CineMatch

GOLD_ML_COLLABORATIVE_DIR:
D:\INSTITUTO CONTINENTAL\5. CICLO - 5\PROYECTO PRODUCTIVO IIIA\PP_CineMatch\data\gold\ml\collaborative



## 3. CARGA DE COLLABORATIVE_INTERACTIONS

In [20]:
collaborative = pd.read_parquet(
    GOLD_ML_COLLABORATIVE_DIR
    / "collaborative_interactions"
)

print(
    "Shape:",
    collaborative.shape
)

print(
    "\nColumnas:",
    collaborative.columns.tolist()
)

collaborative.head()

Shape: (32000204, 4)

Columnas: ['user_id', 'movie_id', 'rating', 'rated_at']


,user_id,movie_id,rating,rated_at
0,1,17,4.0,1999-12-03 19:24:37+00:00
1,1,25,1.0,1999-12-03 19:43:48+00:00
2,1,29,2.0,1999-11-22 00:36:16+00:00
3,1,30,5.0,1999-12-03 19:24:37+00:00
4,1,32,5.0,1999-11-22 00:00:58+00:00


  
## 4. REVISION BASICA

In [21]:
print(
    "Usuarios:",
    collaborative["user_id"].nunique()
)

print(
    "Películas:",
    collaborative["movie_id"].nunique()
)

print(
    "Ratings:",
    len(collaborative)
)

print("\nDistribución de ratings:")

display(
    collaborative["rating"].describe()
)

Usuarios: 200948
Películas: 84432
Ratings: 32000204

Distribución de ratings:


count    3.200020e+07
mean     3.540396e+00
std      1.058986e+00
min      5.000000e-01
25%      3.000000e+00
50%      3.500000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64

## 5. VERIFICAR NULOS

Verificamos y anulamos calquier dato invalido

In [22]:
collaborative[
    [
        "user_id",
        "movie_id",
        "rating",
    ]
].isna().sum()

collaborative = (
    collaborative
    .dropna(
        subset=[
            "user_id",
            "movie_id",
            "rating",
        ]
    )
    .copy()
)


## 6. SELECCIONAMOS USUARIOS ACTIVOS

No se entrenara con los 32mil, se filtrara a los usuarios ocn el historial suficiente

In [24]:
MIN_RATINGS_USER = 20

user_activity = (
    collaborative
    .groupby("user_id")
    .size()
)

active_users = user_activity[
    user_activity >= MIN_RATINGS_USER
].index

collaborative_active = (
    collaborative[
        collaborative["user_id"].isin(
            active_users
        )
    ]
    .copy()
)

print(
    "Usuarios activos:",
    collaborative_active[
        "user_id"
    ].nunique()
)

print(
    "Ratings usuarios activos:",
    len(collaborative_active)
)

Usuarios activos: 200948
Ratings usuarios activos: 32000204


## 7. LIMITAR USUARIOS PARA EL DESARROLLO

In [25]:
MAX_USERS = 10_000

sample_users = (
    collaborative_active["user_id"]
    .drop_duplicates()
    .head(MAX_USERS)
)

collaborative_sample = (
    collaborative_active[
        collaborative_active[
            "user_id"
        ].isin(sample_users)
    ]
    .copy()
)

print(
    "Shape muestra:",
    collaborative_sample.shape
)

print(
    "Usuarios:",
    collaborative_sample[
        "user_id"
    ].nunique()
)

print(
    "Películas:",
    collaborative_sample[
        "movie_id"
    ].nunique()
)

Shape muestra: (1570256, 4)
Usuarios: 10000
Películas: 33493



## 8. CREAMOS UN MAPEO DE USUARIOS Y PELICULAS

In [26]:
user_ids = (
    collaborative_sample["user_id"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

movie_ids = (
    collaborative_sample["movie_id"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

user_to_idx = {
    user_id: idx
    for idx, user_id
    in enumerate(user_ids)
}

movie_to_idx = {
    movie_id: idx
    for idx, movie_id
    in enumerate(movie_ids)
}

idx_to_user = {
    idx: user_id
    for user_id, idx
    in user_to_idx.items()
}

idx_to_movie = {
    idx: movie_id
    for movie_id, idx
    in movie_to_idx.items()
}


## 9. SEPARACION DE DATA TRAIN / TEST  

In [28]:
train_df, test_df = train_test_split(
    collaborative_sample,
    test_size=0.20,
    random_state=42,
)

print(
    "Train:",
    train_df.shape
)

print(
    "Test:",
    test_df.shape
)

Train: (1256204, 4)
Test: (314052, 4)



## 10. CONTRUIR MATRIZ USUARIO - PELICULA

In [29]:
train_user_idx = (
    train_df["user_id"]
    .map(user_to_idx)
    .to_numpy()
)

train_movie_idx = (
    train_df["movie_id"]
    .map(movie_to_idx)
    .to_numpy()
)

train_ratings = (
    train_df["rating"]
    .astype(float)
    .to_numpy()
)

user_movie_matrix = csr_matrix(
    (
        train_ratings,
        (
            train_user_idx,
            train_movie_idx,
        ),
    ),
    shape=(
        len(user_ids),
        len(movie_ids),
    ),
)

print(
    "Shape matriz:",
    user_movie_matrix.shape
)

print(
    "Ratings almacenados:",
    user_movie_matrix.nnz
)

Shape matriz: (10000, 33493)
Ratings almacenados: 1256204



## 11. CALCULAR SPARSITY  

In [30]:
total_positions = (
    user_movie_matrix.shape[0]
    * user_movie_matrix.shape[1]
)

sparsity = (
    1
    - (
        user_movie_matrix.nnz
        / total_positions
    )
)

print(
    "Sparsity:",
    round(
        sparsity * 100,
        4
    ),
    "%"
)

Sparsity: 99.6249 %



## 12. ENTRENAR TRUNCATED SVD

In [32]:
N_FACTORS = 50

svd = TruncatedSVD(
    n_components=N_FACTORS,
    random_state=42,
)

user_factors = svd.fit_transform(
    user_movie_matrix
)

movie_factors = (
    svd.components_.T
)

print(
    "User factors:",
    user_factors.shape
)

print(
    "Movie factors:",
    movie_factors.shape
)




User factors: (10000, 50)
Movie factors: (33493, 50)


## 13. VARIANZA EXPLICADA

In [34]:
explained_variance = (
    svd.explained_variance_ratio_.sum()
)

print(
    "Varianza explicada:",
    round(
        explained_variance,
        4
    )
)

Varianza explicada: 0.3027


## 14. PREPARAR UN TEST VALIDO

In [37]:
train_users = set(
    train_df["user_id"].unique()
)

train_movies = set(
    train_df["movie_id"].unique()
)

test_valid = (
    test_df[
        test_df["user_id"].isin(
            train_users
        )
        &
        test_df["movie_id"].isin(
            train_movies
        )
    ]
    .copy()
)

print(
    "Test original:",
    len(test_df)
)

print(
    "Test evaluable:",
    len(test_valid)
)

Test original: 314052
Test evaluable: 311159


## 15. PREDECIR RATINGS DEL TEST

In [38]:
test_user_idx = (
    test_valid["user_id"]
    .map(user_to_idx)
    .to_numpy()
)

test_movie_idx = (
    test_valid["movie_id"]
    .map(movie_to_idx)
    .to_numpy()
)

predicted_ratings = np.sum(
    user_factors[test_user_idx]
    *
    movie_factors[test_movie_idx],
    axis=1,
)

predicted_ratings = np.clip(
    predicted_ratings,
    0.5,
    5.0,
)

## 16. EVALUAR EL MAE

In [39]:
mae = mean_absolute_error(
    test_valid["rating"],
    predicted_ratings,
)

print(
    "MAE:",
    round(mae, 4)
)

MAE: 2.4718


## 17. EVALUAR EL RMSE

In [40]:
rmse = np.sqrt(
    mean_squared_error(
        test_valid["rating"],
        predicted_ratings,
    )
)

print(
    "RMSE:",
    round(rmse, 4)
)

RMSE: 2.7167


## 18. DATAFRAME DE RESULTADOS

In [42]:
evaluation_df = (
    test_valid[
        [
            "user_id",
            "movie_id",
            "rating",
        ]
    ]
    .copy()
)

evaluation_df[
    "predicted_rating"
] = predicted_ratings

evaluation_df[
    "absolute_error"
] = abs(
    evaluation_df["rating"]
    -
    evaluation_df["predicted_rating"]
)

evaluation_df.head(20)

,user_id,movie_id,rating,predicted_rating,absolute_error
1072652,6945,1617,3.0,1.423255,1.576745
1382836,8821,2710,3.0,0.641932,2.358068
133963,864,5952,4.0,1.222034,2.777966
540429,3503,592,3.5,1.179221,2.320779
196365,1278,196,2.0,1.702904,0.297096
1223186,7883,1210,1.0,0.500000,0.500000
1234441,7940,2412,3.0,0.500000,2.500000
346933,2275,104,3.0,0.972036,2.027964
389867,2578,49272,3.5,0.680162,2.819838
404370,2693,3061,4.0,0.500000,3.500000


## 19. CARGAR DIM_MOVIE PARA MOSTRAR TITULOS

In [43]:
dim_movie = pd.read_parquet(
    GOLD_DIMENSIONS_DIR
    / "dim_movie.parquet"
)

## 20. FUNCION TOP-N

In [44]:
def recommend_top_n(
    user_id,
    n=10,
):
    """
    Genera recomendaciones Top-N
    para un usuario.
    """

    if user_id not in user_to_idx:
        raise ValueError(
            "Usuario no encontrado en el modelo."
        )

    user_idx = user_to_idx[
        user_id
    ]

    # Score para todas las películas
    scores = (
        user_factors[user_idx]
        @ movie_factors.T
    )

    # Películas ya vistas/calificadas
    seen_movies = set(
        train_df.loc[
            train_df["user_id"] == user_id,
            "movie_id",
        ]
    )

    candidates = []

    for movie_idx, score in enumerate(
        scores
    ):

        movie_id = idx_to_movie[
            movie_idx
        ]

        if movie_id in seen_movies:
            continue

        candidates.append(
            (
                movie_id,
                float(score),
            )
        )

    candidates = sorted(
        candidates,
        key=lambda x: x[1],
        reverse=True,
    )

    top_n = candidates[:n]

    recommendations = pd.DataFrame(
        top_n,
        columns=[
            "movie_id",
            "predicted_score",
        ],
    )

    recommendations[
        "predicted_score"
    ] = recommendations[
        "predicted_score"
    ].clip(
        lower=0.5,
        upper=5.0,
    )

    recommendations = (
        recommendations.merge(
            dim_movie[
                [
                    "movie_id",
                    "title",
                ]
            ],
            on="movie_id",
            how="left",
        )
    )

    return recommendations[
        [
            "movie_id",
            "title",
            "predicted_score",
        ]
    ]

## 21. PROBAR EL TOP-3

In [46]:
usuario_prueba = (
    train_df["user_id"]
    .iloc[0]
)

print(
    "Usuario:",
    usuario_prueba
)

top_3 = recommend_top_n(
    user_id=usuario_prueba,
    n=3,
)

top_3

Usuario: 368


,movie_id,title,predicted_score
0,3793,X-Men (2000),3.378254
1,7153,"Lord of the Rings: The Return of the King, The...",3.143790
2,1584,Contact (1997),3.109216


## 22. CONVERTIR SCORE A MATCH %

In [47]:
top_3["match_percent"] = (
    top_3["predicted_score"]
    / 5.0
    * 100
)

top_3[
    "match_percent"
] = (
    top_3[
        "match_percent"
    ]
    .round(1)
)

top_3[
    [
        "title",
        "predicted_score",
        "match_percent",
    ]
]

,title,predicted_score,match_percent
0,X-Men (2000),3.378254,67.6
1,"Lord of the Rings: The Return of the King, The...",3.143790,62.9
2,Contact (1997),3.109216,62.2


## Conclusiones

Se implementó un primer modelo colaborativo mediante factorización de matriz con TruncatedSVD.

El modelo aprende factores latentes de usuarios y películas utilizando las calificaciones históricas de MovieLens.

La evaluación mediante MAE y RMSE permite cuantificar el error de predicción de ratings, mientras que la función Top-N transforma las predicciones en recomendaciones personalizadas excluyendo contenidos previamente evaluados por el usuario.

Este modelo servirá como componente colaborativo del futuro sistema híbrido de CineMatch.